:::{admonition} Download
:class: important

Download this notebook: **{nb-download}`save_and_load.ipynb`**!

:::


# Saving and Loading

## Saving and Loading a Model

In nemos, you can save a model by calling the {py:meth}`~nemos.glm.GLM.save_params` method, which writes a {py:func}`npz file <numpy.savez>`, a NumPy-specific binary format.

In [1]:
import nemos as nmo

# define a ridge regularized glm, with LBFGS solver
model = nmo.glm.GLM(
    regularizer="Ridge",
    solver_name="LBFGS"
)

# save
model.save_params("ridge_glm_params.npz")


# load
loaded_model = nmo.load_model("ridge_glm_params.npz")

print("Original Model: \n", model)
print("\nLoaded Model: \n", loaded_model)


Original Model: 
 GLM(
    observation_model=PoissonObservations(),
    inverse_link_function=exp,
    regularizer=Ridge(),
    regularizer_strength=1.0,
    solver_name='LBFGS'
)

Loaded Model: 
 GLM(
    observation_model=PoissonObservations(),
    inverse_link_function=exp,
    regularizer=Ridge(),
    regularizer_strength=1.0,
    solver_name='LBFGS'
)


## Saving and Loading a Fitted Model

The same workflow works for fitted models, meaning the learned coefficients and intercepts are also saved and restored:

In [2]:
import numpy as np

# generate some data
np.random.seed(123)
X, weights = np.random.randn(50, 1), 0.1 * np.random.randn(1)
counts = np.random.poisson(np.exp(X @ weights))

# fit and save
model.fit(X, counts)
model.save_params("ridge_glm_params_fitted.npz")

# load
loaded_model = nmo.load_model("ridge_glm_params_fitted.npz")

print("Original coefficient and intercept:", model.coef_, model.intercept_)
print("Loaded coefficient and intercept:", loaded_model.coef_, loaded_model.intercept_)

Original coefficient and intercept: [-0.00710396] [0.05829372]
Loaded coefficient and intercept: [-0.00710396] [0.05829372]


## Inspecting the `npz`

You can inspect the contents of a saved `.npz` file with {py:func}`~nemos.io.inspect_npz`, which displays the stored metadata and parameter keys—useful for debugging (e.g., when loading fails) or verifying saved models.

In [3]:
nmo.inspect_npz("ridge_glm_params.npz")

Metadata
--------
jax version            : 0.11.1 (installed: 0.11.1)
jaxlib version         : 0.11.1 (installed: 0.11.1)
scipy version          : 1.18.0 (installed: 1.18.0)
scikit-learn version   : 1.9.0 (installed: 1.9.0)
nemos version          : 0.2.10.dev379 (installed: 0.2.10.dev379)

Model class
-----------
Saved model class      : nemos.glm.glm.GLM

Model parameters
----------------
inverse_link_function  : jax.numpy.exp
observation_model      : {'class': 'nemos.observation_models.PoissonObservations'}
regularizer            : {'class': 'nemos.regularizer.Ridge'}
regularizer_strength   : 1
solver_kwargs          : None
solver_name            : LBFGS

Model fit parameters
--------------------
aux_: None
coef_: None
dof_resid_: None
intercept_: None
scale_: None


## Save and Load Custom Objects

Advanced users may want to specify custom models and still be able to save and load. For example, one could try a different inverse link function (non-linearity) or a custom  `Regularizer`.

In [4]:
def custom_link(x):
    return x**2

class CustomRegularizer(nmo.regularizer.Ridge):
    def __init__(self, new_param):
        self.new_param = new_param

model = nmo.glm.GLM(inverse_link_function=custom_link, regularizer=CustomRegularizer(10))
model.save_params("custom_regularizer_params.npz")

nmo.inspect_npz("custom_regularizer_params.npz")

Metadata
--------
jax version            : 0.11.1 (installed: 0.11.1)
jaxlib version         : 0.11.1 (installed: 0.11.1)
scipy version          : 1.18.0 (installed: 1.18.0)
scikit-learn version   : 1.9.0 (installed: 1.9.0)
nemos version          : 0.2.10.dev379 (installed: 0.2.10.dev379)

Model class
-----------
Saved model class      : nemos.glm.glm.GLM

Model parameters
----------------
inverse_link_function  : __main__.custom_link
observation_model      : {'class': 'nemos.observation_models.PoissonObservations'}
regularizer            : {'class': '__main__.CustomRegularizer', 'params': {'new_param': 10}}
regularizer_strength   : 1
solver_kwargs          : None
solver_name            : Newton

Model fit parameters
--------------------
aux_: None
coef_: None
dof_resid_: None
intercept_: None
scale_: None


As you can see, the regularizer class is stored as a string, `"{object_class.__module__}.{object_class.__name__}"`. This means that trying to load this model directly will result in an error, because NeMoS doesn’t pickle objects and therefore doesn’t know how to recreate the `CustomRegularizer` automatically.

:::{admonition} Why prevent pickling?
:class: warning

Unpickling typically involves executing code, which can pose a security risk.
A third party could tamper with a pickled file to insert malicious code that runs whenever the object is unpickled.

For a real-world example, see [this discussion](https://news.ycombinator.com/item?id=41901475).
:::

In [5]:
loaded_model = nmo.load_model("custom_regularizer_params.npz")

ValueError: The class 'CustomRegularizer' is not a native NeMoS class.
To load a custom regularization class, please provide the following mapping:

 - nemos.load_model(save_path, mapping_dict={'regularizer': CustomRegularizer})

As the error explains, you can tell nemos how to load the custom objects by providing a mapping between the saved string and to the callable.

In [6]:
mapping = {
    "regularizer": CustomRegularizer,
    "inverse_link_function": custom_link
}
loaded_model = nmo.load_model("custom_regularizer_params.npz", mapping_dict=mapping)
loaded_model

/home/jenkins/agent/workspace/CCN_nemos_PR-612/venv/lib/python3.12/site-packages/nemos/io/io.py:160: UserWarning: The following keys have been replaced in the model parameters: ['inverse_link_function', 'regularizer'].
  warnings.warn(


,observation_model,PoissonObservations()
,inverse_link_function,<function cus...x7f09f5354860>
,regularizer,CustomRegular...(new_param=10)
,regularizer_strength,1.0
,solver_name,'Newton'
,solver_kwargs,{}
,regularizer__new_param,10
Name,Type,Value
aux_,NoneType,None
coef_,NoneType,None
dof_resid_,NoneType,None


:::{admonition} Allowed Mappings
:class: warning

Mapping is allowed **only** for callables (functions) and classes, because these cannot be stored directly without pickling.
Other values (like numbers, strings, or arrays) are always stored directly in the `.npz` and cannot be remapped.

When mapping a custom class, you must pass the **class itself** (e.g., `mapping = {"regularizer": CustomRegularizer}`), not an instance (`CustomRegularizer()`).
Passing an instance would overwrite the saved parameters and could lead to inconsistencies.

:::